# Watermark Algorithm Verification

Run this first — no GPU or model download needed (~5 seconds).

Verifies that `WatermarkLogitsProcessor` and `WatermarkDetector` produce identical green lists from the same hash seed. If this passes, the core algorithm is correct and all downstream experiments will be valid.

In [ ]:
import os, sys

GITHUB_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
BRANCH     = 'AmmarChanges'
REPO_DIR   = 'llm-invisible-watermarking'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {BRANCH}...')
    os.system(f'git clone --branch {BRANCH} {GITHUB_URL} {REPO_DIR}')
else:
    print('Pulling latest...')
    os.system(f'cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q transformers torch scipy numpy')
print('✓ Minimal packages installed')

In [ ]:
import random, torch
from watermark.logits_processor import WatermarkLogitsProcessor
from watermark.detector import WatermarkDetector

proc = WatermarkLogitsProcessor(vocab_size=32000, gamma=0.5, seed=42)
det  = WatermarkDetector(vocab_size=32000, gamma=0.5, seed=42)

# Green lists must match for every token
for tok in [0, 1, 1234, 31999]:
    p = set(proc._get_greenlist_ids(tok).tolist())
    d = det._get_greenlist_set(tok)
    assert p == d, f'HASH MISMATCH at token {tok}'
print('✓ Processor and detector produce identical green lists')

# Delta applied to green logits only
scores = torch.zeros(1, 32000)
mod    = proc(torch.tensor([[1234]]), scores.clone())
green  = proc._get_greenlist_ids(1234).tolist()
assert all(mod[0, g].item() > 0 for g in green[:10])
assert mod[0, 0].item() == 0.0
print('✓ Delta applied to green logits only')

# Forced-green sequence → high z-score
random.seed(999)
toks = [random.randint(0, 31999)]
for _ in range(200):
    toks.append(random.choice(proc._get_greenlist_ids(toks[-1]).tolist()))
r = det.score_sequence(toks)
print(f'✓ Forced-green sequence: z={r.z_score:.2f}  green={r.green_fraction:.0%}')
assert r.z_score > 10

# Random sequence → not detected
null = [random.randint(0, 31999) for _ in range(200)]
nr   = det.score_sequence(null)
print(f'✓ Random sequence:       z={nr.z_score:.2f}  green={nr.green_fraction:.0%}')
assert abs(nr.z_score) < 4

print('\n✅  Algorithm verification PASSED — safe to run LLaMA and Gemma notebooks')